# Aula 07 — Schemas de estado: o contrato do dict compartilhado

**Objetivo:** saber escolher entre `TypedDict`, `dataclass` e `Pydantic` — e
entender de uma vez por que `mood='bravo'` passa calado nos dois primeiros e
é barrado no terceiro.

**Pré-requisito:** Aula 02 (o estado nasce do `invoke`; cada chave é um canal).

**Como usar:** rode em ordem. Autocontido, 100% offline (sem LLM, sem chave).
Comece pela CÉLULA 0.

In [ ]:
# CÉLULA 0 — instalar dependências
# No Colab esta célula é necessária. Localmente (uv sync feito) pode pular.
%pip install -q langgraph pydantic
print('dependências prontas.')

## 1. Analogia: o formulário da lanchonete tem 3 versões

O campo `humor do cliente` só aceita `feliz` ou `triste`. Três recepcionistas:

| Contrato | Recepcionista | Comportamento com `humor='bravo'` |
|---|---|---|
| `TypedDict` | rascunho a lápis | anota sem ler — dica de tipo, zero fiscalização |
| `dataclass` | ficha com acesso por ponto (`ficha.nome`) | organiza, mas também não fiscaliza |
| `Pydantic` | recepcionista com carimbo | **barra na entrada** com `ValidationError` |

Tradução técnica: `TypedDict` e `dataclass` dão **hint estático** (seu editor
sublinha, o Python em execução ignora); Pydantic **valida em runtime**.
`Literal['feliz', 'triste']` é o 'enum' que diz os valores permitidos — mas só
o Pydantic o impõe de verdade.

In [ ]:
# CÉLULA 1 — os 3 contratos lado a lado (100% offline)
from dataclasses import dataclass
from typing import Literal, TypedDict
from pydantic import BaseModel, field_validator

class TDState(TypedDict):
    name: str
    mood: Literal['feliz', 'triste']

@dataclass
class DCState:
    name: str
    mood: Literal['feliz', 'triste']

class PYState(BaseModel):
    name: str
    mood: Literal['feliz', 'triste']

    @field_validator('mood')
    @classmethod
    def checa_humor(cls, valor: str) -> str:
        return valor.strip().lower()   # regra EXTRA (o Literal já barrou antes)

print('TD:', TDState(name='Zé', mood='feliz'))
print('DC:', DCState(name='Zé', mood='feliz'))
print('PY:', PYState(name='Zé', mood='feliz'))

## 2. Conceito: hint estático vs validação em runtime

- **Hint estático** (`TypedDict`, `dataclass`, `Literal`): o editor/IDE avisa,
  o `mypy` acusa — mas o Python rodando **não verifica nada**. `mood='bravo'`
  entra, circula no grafo e sai, sem ninguém chiar.
- **Validação em runtime** (Pydantic): o objeto **nasce validado**. Dado ruim
  nem vira instância: morre no construtor com `ValidationError`.
- **`Literal`** sozinho não barra nada fora do Pydantic: em `TypedDict` e
  `dataclass` ele é tinta no papel.
- **Ordem no Pydantic**: o `Literal` valida pertinência primeiro; o
  `@field_validator` roda depois — use-o para regras extras (normalizar caixa,
  aparar espaços, validar formato), não para repetir o `Literal`.

In [ ]:
# CÉLULA 2 — o mesmo 'bravo' nos 3 contratos
from pydantic import ValidationError

print('TypedDict:', TDState(name='Zé', mood='bravo'), '<- entrou calado')
print('dataclass:', DCState(name='Zé', mood='bravo'), '<- entrou calado')
try:
    PYState(name='Zé', mood='bravo')
    print('Pydantic: entrou?! (não deveria)')
except ValidationError as erro:
    print('Pydantic: barrado na construção.')
    print('motivo:', str(erro).splitlines()[2].strip())

## 3. Dissecando o acesso: `state['x']` vs `state.x`

- Grafo `TypedDict`: o nó recebe um **dict** → lê com `state['name']`.
- Grafo `dataclass`/`Pydantic`: o nó recebe a **instância** (`DCState`,
  `PYState`) → lê com `state.name`.
- O `invoke` acompanha: `TypedDict` recebe dict; `dataclass`/`Pydantic` aceitam
  a instância (e o `dataclass` aceita dict também).
- O **retorno do nó é sempre dict parcial** nos 3 casos (o Pydantic até aceita
  instância no retorno, mas dict é o padrão).
- O **resultado final do `invoke` é sempre dict**, mesmo no grafo Pydantic.

In [ ]:
# CÉLULA 3 — o mesmo grafo, 3 acessos (rode e compare)
import random
from langgraph.graph import START, END, StateGraph

random.seed(1)

def td_no1(state: TDState) -> dict:
    return {'name': state['name'] + ' está ... '}       # dict: colchetes

def dc_no1(state: DCState) -> dict:
    return {'name': state.name + ' está ... '}          # instância: ponto

def py_no1(state: PYState) -> dict:
    return {'name': state.name + ' está ... '}          # instância: ponto

def monta(schema, no1):
    b = StateGraph(schema)
    b.add_node('node_1', no1)
    b.add_node('node_2', lambda s: {'mood': 'feliz'})
    b.add_node('node_3', lambda s: {'mood': 'triste'})
    b.add_edge(START, 'node_1')
    b.add_conditional_edges('node_1', lambda s: 'node_2')
    b.add_edge('node_2', END); b.add_edge('node_3', END)
    return b.compile()

print('TD:', monta(TDState, td_no1).invoke({'name': 'Zé', 'mood': 'triste'}))
print('DC:', monta(DCState, dc_no1).invoke(DCState(name='Zé', mood='triste')))
print('PY:', monta(PYState, py_no1).invoke(PYState(name='Zé', mood='triste')))

## 4. Variações que quebram (todas executadas e verificadas)

### 4a. Chave extra no Pydantic: ignorada em silêncio

Por padrão, campo desconhecido não dá erro — é descartado. (Dá para endurecer
com `model_config = ConfigDict(extra='forbid')`, mas o padrão é ignorar.)

In [ ]:
# CÉLULA 4 — chave extra (100% offline)
out = monta(PYState, py_no1).invoke({'name': 'Zé', 'mood': 'feliz', 'extra': 1})
print(out)
assert 'extra' not in out
print('OK — campo desconhecido descartado, sem erro.')

### 4b. Validador custom: para normalizar, não para repetir o `Literal`

'bravo' nem chega ao `@field_validator` — o `Literal` barra antes, com erro
dele. O custom brilha normalizando entrada válida e suja (`'  FELIZ '` →
`'feliz'`).

In [ ]:
# CÉLULA 5 — normalizador (100% offline)
from pydantic import BaseModel, field_validator

class HumorLimpo(BaseModel):
    mood: str

    @field_validator('mood')
    @classmethod
    def limpa(cls, v: str) -> str:
        return v.strip().lower()

print(repr(HumorLimpo(mood='  FELIZ ').mood))
assert HumorLimpo(mood='  FELIZ ').mood == 'feliz'
print('OK — validador custom normaliza o que o Literal deixaria passar sujo.')

### 4c. `dataclass` aceita dict no `invoke` — e o inválido persiste

Sem nó que sobrescreva, `mood='bravo'` atravessa o grafo inteiro e sai
intacto. É a prova de que `dataclass` não fiscaliza nada em runtime.

In [ ]:
# CÉLULA 6 — bravo atravessa o grafo dataclass (100% offline)
b = StateGraph(DCState)
b.add_node('passa', lambda s: {})
b.add_edge(START, 'passa'); b.add_edge('passa', END)
out = b.compile().invoke(DCState(name='Zé', mood='bravo'))
print(out)
assert out['mood'] == 'bravo'
print('OK — entrou, circulou e saiu. Silêncio total.')

## 5. Erros clássicos

1. **Achar que `TypedDict` valida.** Ele documenta; quem barra é o Pydantic
   (CÉLULA 2). `Literal` fora do Pydantic é tinta no papel.
2. **Misturar os acessos.** `state['name']` num grafo dataclass → `TypeError`
   (instância não é subscritível); `state.name` num grafo TypedDict →
   `AttributeError`. O acesso segue o schema.
3. **Esperar o `@field_validator` barrar o que o `Literal` já barra.** A ordem
   é Literal primeiro; o custom nunca vê o inválido (CÉLULA 5 mostra o uso certo).
4. **Achar que o resultado final é instância.** `invoke` em grafo Pydantic
   devolve **dict** — a instância só existe *dentro* da execução.
5. **Esquecer campo obrigatório no `invoke`.** Sem valor inicial, o 1º nó que
   ler estoura `KeyError` (Aula 02, seção 5d) — em qualquer dos 3 schemas.
6. **Validar no lugar errado.** Regra de negócio que muda ('nota ≥ 7') pertence
   ao **nó** ou ao validador; formato ('nota é int 0–10') pertence ao **schema**.
   Misturar os dois espalha a regra.

## 6. Exercício guiado — `Resultado` nos 3 schemas

Situação: o avaliador de nota da Aula 02 (`aluno`, `nota`, `status`) agora com
contrato explícito. Monte o `Resultado` nos 3 schemas e rode o mesmo grafo
(`calcula`: nota ≥ 7 → `aprovado`, senão `reprovado`).

In [ ]:
# CÉLULA 7 — exercício guiado (rode e compare os 3)
from dataclasses import dataclass as _dc
from typing import TypedDict as _TD
from pydantic import BaseModel as _BM

class R_TD(_TD):
    aluno: str; nota: int; status: str

@_dc
class R_DC:
    aluno: str; nota: int; status: str

class R_PY(_BM):
    aluno: str; nota: int; status: str

def monta_resultado(schema):
    b = StateGraph(schema)
    b.add_node('ok', lambda s: {'status': 'aprovado'})
    b.add_edge(START, 'ok'); b.add_edge('ok', END)
    return b.compile()

print('TD:', monta_resultado(R_TD).invoke({'aluno': 'Ana', 'nota': 8, 'status': ''}))
print('DC:', monta_resultado(R_DC).invoke(R_DC(aluno='Ana', nota=8, status='')))
print('PY:', monta_resultado(R_PY).invoke(R_PY(aluno='Ana', nota=8, status='')))
print('OK — mesmo resultado, 3 contratos. O cálculo real vem no desafio.')

## 7. Desafio — Pydantic que barra nota absurda e normaliza status

Monte `ResultadoPY` com:

- `nota: int` com validador que **barra** fora de 0–10 (`ValueError` se
  `nota=15` na construção);
- `status: str` com validador que **normaliza** (`' APROVADO '` → `'aprovado'`);
- um grafo `calcula` que lê `nota` e grava `status` (`≥ 7 → aprovado`).

Valide: `ResultadoPY(aluno='Ana', nota=8, status='')` passa; `nota=15` explode
na construção (antes do grafo rodar); `' APROVADO '` sai `'aprovado'`. É o
padrão do seu avaliador de redação: schema barra formato, nó decide negócio.

## 8. Checklist + próxima aula

**Você entendeu se consegue responder sem olhar:**
- [ ] O que cada schema fiscaliza em runtime — e qual não fiscaliza nada?
- [ ] Como o nó lê o estado em cada schema (`['x']` vs `.x`)?
- [ ] Onde o `ValidationError` acontece: no `invoke`, na construção, ou nos dois?
- [ ] Para que serve o `@field_validator` se o `Literal` já barra?
- [ ] O que o `invoke` devolve num grafo Pydantic: dict ou instância?
- [ ] Regra de formato vs regra de negócio: onde mora cada uma?

**Próxima aula (08 — Reducers):** `overwrite` vs `operator.add`, reducer custom,
fan-out paralelo que quebra sem reducer, e `RemoveMessage` — a poda do histórico.